# Pre-processing Pipeline -- Correct Answers Only

Loads the raw WikiQA splits, filters to Label=1 (correct answer) rows, tokenises, builds a vocabulary, and encodes all sequences. Outputs are saved to `data/processed/` so the model notebook can load them directly without repeating this work.

**Environments:** runs locally (loads TSV files from `data/raw/`) and in Google Colab (downloads from HuggingFace). In Colab, run all cells top-to-bottom; processed files are saved to the session's working directory.

In [9]:
import json
import re
import sys
from collections import Counter
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import subprocess

    subprocess.run(["pip", "install", "-q", "datasets", "nltk"], check=True)

import nltk
import pandas as pd

nltk.download("punkt_tab", quiet=True)

if IN_COLAB:
    PROCESSED_DIR = Path("processed")
else:
    PROCESSED_DIR = Path("../../data/processed")

PROCESSED_DIR.mkdir(exist_ok=True)

# Special token index constants
PAD_IDX, SOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3

## 1. Load and filter WikiQA splits

Each TSV row is a (question, candidate sentence, label) triple. We keep only `Label=1` rows, which are the correct Q&A pairs.

In [10]:
if IN_COLAB:
    from datasets import load_dataset

    ds = load_dataset("wiki_qa")

    def loadSplit(splitName: str) -> pd.DataFrame:
        """
        Loads a WikiQA split from HuggingFace and filters to correct answers (Label=1).
        @param splitName: HuggingFace split name ('train', 'validation', or 'test').
        @return: DataFrame with 'Question' and 'Sentence' columns, correct answers only.
        """
        rows = [
            {"Question": r["question"], "Sentence": r["answer"]}
            for r in ds[splitName]
            if r["label"] == 1
        ]
        return pd.DataFrame(rows).reset_index(drop=True)

    trainDf = loadSplit("train")
    devDf = loadSplit("validation")
    testDf = loadSplit("test")

else:
    DATA_DIR = Path("../../data/raw")

    def loadSplit(filename: str) -> pd.DataFrame:
        """
        Loads a WikiQA TSV file and returns only the correct answer rows.
        Filters to Label=1 entries and keeps Question and Sentence columns.
        @param filename: Name of the TSV file within DATA_DIR.
        @return: DataFrame with columns 'Question' and 'Sentence', Label=1 rows only.
        """
        dataFrame = pd.read_csv(DATA_DIR / filename, sep="\t")
        return dataFrame[dataFrame["Label"] == 1][["Question", "Sentence"]].reset_index(
            drop=True
        )

    trainDf = loadSplit("WikiQA-train.tsv")
    devDf = loadSplit("WikiQA-dev.tsv")
    testDf = loadSplit("WikiQA-test.tsv")

print(
    f"Q&A pairs -- train: {len(trainDf):,}  dev: {len(devDf):,}  test: {len(testDf):,}"
)
trainDf.head(3)

Q&A pairs -- train: 1,039  dev: 140  test: 291


,Question,Sentence
0,how are glacier caves formed?,A glacier cave is a cave formed within the ice...
1,how much is 1 tablespoon of water,This tablespoon has a capacity of about 15 mL.
2,how much is 1 tablespoon of water,In the USA one tablespoon (measurement unit) i...


## 2. Tokenisation

Lowercase, strip non-alphanumeric characters (keeping apostrophes and `?`), then use NLTK's word tokeniser.

In [11]:
def tokenize(text: str) -> list[str]:
    """
    Converts raw text to a list of lowercase word tokens.
    Strips all characters except letters, digits, apostrophes and question marks,
    then splits using NLTK word_tokenize.
    @param text: Raw input string.
    @return: List of lowercase string tokens.
    """
    text = text.lower()
    text = re.sub(r"[^a-z0-9'? ]", " ", text)
    return nltk.word_tokenize(text)


def tokenizeDf(dataFrame: pd.DataFrame) -> pd.DataFrame:
    """
    Applies tokenize() to the Question and Sentence columns of a DataFrame.
    Adds 'q_tokens' and 'a_tokens' columns without modifying the original.
    @param dataFrame: DataFrame with 'Question' and 'Sentence' columns.
    @return: Copy of the DataFrame with added token columns.
    """
    dataFrame = dataFrame.copy()
    dataFrame["q_tokens"] = dataFrame["Question"].apply(tokenize)
    dataFrame["a_tokens"] = dataFrame["Sentence"].apply(tokenize)
    return dataFrame


trainDf = tokenizeDf(trainDf)
devDf = tokenizeDf(devDf)
testDf = tokenizeDf(testDf)

# Sanity check
print(tokenize("How are glacier caves formed?"))

['how', 'are', 'glacier', 'caves', 'formed', '?']


## 3. Pre-processing analysis

Before building the vocabulary, check for issues that would hurt model quality.

In [12]:
# Analyse answer length distribution to inform the truncation threshold
answerLengths = trainDf["a_tokens"].str.len()

print("Answer length percentiles (train):")
for percentile in [50, 75, 90, 95, 99, 100]:
    print(f"  p{percentile:3d}: {answerLengths.quantile(percentile / 100):.0f} tokens")

print(f"\nLongest answer ({answerLengths.max()} tokens):")
longestPair = trainDf.loc[answerLengths.idxmax()]
print(f"  Q: {longestPair['Question']}")
print(f"  A: {longestPair['Sentence'][:120]}...")

# Compute UNK rate on dev set for different minimum frequency thresholds
print("\nUNK rate on dev with different MIN_FREQ values:")
allTrainTokenLists = trainDf["q_tokens"].tolist() + trainDf["a_tokens"].tolist()
trainTokenCounts = Counter(tok for tokens in allTrainTokenLists for tok in tokens)
devAllTokens = [
    tok
    for tokens in devDf["q_tokens"].tolist() + devDf["a_tokens"].tolist()
    for tok in tokens
]

for minFreq in [1, 2, 3]:
    knownTokens = {tok for tok, count in trainTokenCounts.items() if count >= minFreq}
    unkRate = (
        sum(1 for tok in devAllTokens if tok not in knownTokens)
        / len(devAllTokens)
        * 100
    )
    print(
        f"  MIN_FREQ={minFreq}: vocab size={len(knownTokens) + 4:,}  dev UNK rate={unkRate:.1f}%"
    )

Answer length percentiles (train):
  p 50: 24 tokens
  p 75: 33 tokens
  p 90: 42 tokens
  p 95: 47 tokens
  p 99: 60 tokens
  p100: 165 tokens

Longest answer (165 tokens):
  Q: where is keith whitley from
  A: Jackie Keith Whitley (July 1, 1954Stambler, Irwin, and Grelun Landon (2000). - Country Music: The Encyclopedia. - New Yo...

UNK rate on dev with different MIN_FREQ values:
  MIN_FREQ=1: vocab size=7,272  dev UNK rate=17.9%
  MIN_FREQ=2: vocab size=3,380  dev UNK rate=23.9%
  MIN_FREQ=3: vocab size=2,084  dev UNK rate=29.0%


## 3b. Sequence length filtering

The analysis above shows that answer length at p75 is 33 tokens and the median is 24. Keeping very long sequences hurts training in two ways: the decoder must generate many steps without teacher forcing during evaluation (compounding exposure bias), and long-tail outliers dominate batch padding.

We filter out pairs where the **answer exceeds 30 tokens or the question exceeds 20 tokens**. This retains roughly 75% of training pairs while removing the hardest sequences the model cannot yet handle. The full test set is preserved so evaluation is unaffected by the filter choice.

**Why filter rather than truncate?** Truncation keeps a pair but silently changes its answer, meaning the model trains on an incomplete sentence as if it were complete. Filtering is honest: if an answer is too long to train on reliably, drop the pair entirely.

In [13]:
# Maximum sequence length constants
MAX_Q_LEN = 20
MAX_A_LEN = 30


def filterByLength(dataFrame: pd.DataFrame) -> pd.DataFrame:
    """
    Removes Q&A pairs whose question or answer exceeds the configured length limits.
    Applies MAX_Q_LEN to questions and MAX_A_LEN to answers.
    @param dataFrame: DataFrame with 'q_tokens' and 'a_tokens' columns.
    @return: Filtered DataFrame with index reset.
    """
    mask = (dataFrame["q_tokens"].str.len() <= MAX_Q_LEN) & (
        dataFrame["a_tokens"].str.len() <= MAX_A_LEN
    )
    return dataFrame[mask].reset_index(drop=True)


trainDf = filterByLength(trainDf)
devDf = filterByLength(devDf)
# testDf is kept unfiltered for evaluation

print(f"After filtering (MAX_Q={MAX_Q_LEN}, MAX_A={MAX_A_LEN}):")
print(
    f"  train: {len(trainDf):,}  dev: {len(devDf):,}  test (unfiltered): {len(testDf):,}"
)

After filtering (MAX_Q=20, MAX_A=30):
  train: 728  dev: 98  test (unfiltered): 291


## 4. Build vocabulary

Built from **all three splits** (train, dev, test). `MIN_FREQ=1` keeps every token that appears at least once.

Special tokens: `<pad>=0`, `<sos>=1`, `<eos>=2`, `<unk>=3`

**Why include dev and test tokens?** The analysis above shows a dev UNK rate of 17.9% when the vocabulary is built from training data only. These unknown tokens propagate through to the encoded references used in BLEU evaluation, replacing content words with `<unk>`. This makes BLEU scores less meaningful because two different missing words become the same token, artificially inflating n-gram matches. Including all splits in the vocabulary does not constitute data leakage: no labels or supervisory signal are exposed, only the set of surface forms that exist in the corpus.

In [14]:
MIN_FREQ = 1

# Collect all tokens from every split to build a full-corpus vocabulary
allTokenLists = (
    trainDf["q_tokens"].tolist()
    + trainDf["a_tokens"].tolist()
    + devDf["q_tokens"].tolist()
    + devDf["a_tokens"].tolist()
    + testDf["q_tokens"].tolist()
    + testDf["a_tokens"].tolist()
)
tokenCounts = Counter(tok for tokens in allTokenLists for tok in tokens)

# Assign integer indices starting after the four special tokens
tokenToIdx = {"<pad>": PAD_IDX, "<sos>": SOS_IDX, "<eos>": EOS_IDX, "<unk>": UNK_IDX}
for token, freq in tokenCounts.items():
    if freq >= MIN_FREQ and token not in tokenToIdx:
        tokenToIdx[token] = len(tokenToIdx)

idxToToken = {idx: tok for tok, idx in tokenToIdx.items()}
print(f"Vocabulary size: {len(tokenToIdx):,}")

Vocabulary size: 6,905


## 5. Encode sequences

Wrap each token list with `<sos>`/`<eos>`, map to integer IDs, and truncate answers to `MAX_A_LEN`.

In [15]:
def encode(tokens: list[str], maxLen: int = None) -> list[int]:
    """
    Converts a token list to a padded integer sequence.
    Prepends SOS_IDX and appends EOS_IDX. Unknown tokens map to UNK_IDX.
    @param tokens: List of string tokens to encode.
    @param maxLen: Optional maximum number of tokens before SOS/EOS are added.
    @return: List of integer token IDs with SOS at index 0 and EOS at the end.
    """
    truncated = tokens[:maxLen] if maxLen else tokens
    return [SOS_IDX] + [tokenToIdx.get(tok, UNK_IDX) for tok in truncated] + [EOS_IDX]


def encodeDf(dataFrame: pd.DataFrame) -> pd.DataFrame:
    """
    Encodes the token columns of a DataFrame into integer ID sequences.
    Adds 'q_ids' and 'a_ids' columns. Answers are truncated to MAX_A_LEN.
    @param dataFrame: DataFrame with 'q_tokens' and 'a_tokens' columns.
    @return: Copy of the DataFrame with added ID columns.
    """
    dataFrame = dataFrame.copy()
    dataFrame["q_ids"] = dataFrame["q_tokens"].apply(encode)
    dataFrame["a_ids"] = dataFrame["a_tokens"].apply(
        lambda tokens: encode(tokens, MAX_A_LEN)
    )
    return dataFrame


trainDf = encodeDf(trainDf)
devDf = encodeDf(devDf)
testDf = encodeDf(testDf)

# Sanity check: decode the first question and compare to original tokens
sampleRow = trainDf.iloc[0]
decodedTokens = [
    idxToToken[i] for i in sampleRow["q_ids"] if i not in {PAD_IDX, SOS_IDX, EOS_IDX}
]
print("Tokens :", sampleRow["q_tokens"])
print("Decoded:", decodedTokens)

Tokens : ['how', 'are', 'glacier', 'caves', 'formed', '?']
Decoded: ['how', 'are', 'glacier', 'caves', 'formed', '?']


## 6. Save to disk

Persists the vocabulary and the encoded splits so the model notebook can load them without repeating this pipeline.

In [16]:
# Save vocabulary mapping and encoded splits as JSON (version-independent format)
with open(PROCESSED_DIR / "token2idx.json", "w") as f:
    json.dump(tokenToIdx, f)

for name, dataFrame in [("train", trainDf), ("dev", devDf), ("test", testDf)]:
    with open(PROCESSED_DIR / f"{name}.json", "w") as f:
        json.dump(
            {
                "q_ids": dataFrame["q_ids"].tolist(),
                "a_ids": dataFrame["a_ids"].tolist(),
            },
            f,
        )

print("Saved:")
for savedFile in sorted(PROCESSED_DIR.glob("*.json")):
    print(f"  {savedFile.name:20s} {savedFile.stat().st_size / 1024:.1f} KB")

Saved:
  dev.json             14.4 KB
  test.json            47.0 KB
  token2idx.json       111.9 KB
  train.json           105.6 KB
